# 74 — Chemprop: CRC + ChEMBL NR Multi-Task (8 targets)

Chemprop MPNN trained on CRC pEC50 (primary) plus 7 ChEMBL NR targets as auxiliary regression tasks. Uses NaN masking so each compound only contributes to targets where it has data.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from pxr.data import load_train, load_test
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, to_inchikey, standardize_smiles
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS

SEED = 42; N_FOLDS = 5


In [2]:
def full_metrics(y_true, y_pred, cliff_pairs_df=None, label=""):
    """RAE, MAE, R², Pearson, Spearman, Kendall, Cliff_accuracy."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[mask], yp[mask]

    mae_v  = float(np.mean(np.abs(yt - yp)))
    rae_v  = mae_v / float(np.mean(np.abs(yt - yt.mean()))) if yt.std() > 0 else float("nan")
    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - yt.mean()) ** 2))
    r2_v   = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    pr_v, _ = stats.pearsonr(yt, yp)
    sp_v, _ = stats.spearmanr(yt, yp)
    kt_v, _ = stats.kendalltau(yt, yp)

    m = dict(RAE=rae_v, MAE=mae_v, R2=r2_v,
             Pearson=pr_v, Spearman=sp_v, Kendall=kt_v)

    if cliff_pairs_df is not None and len(cliff_pairs_df) > 0:
        correct = total = 0
        for _, row in cliff_pairs_df.iterrows():
            ia, ii = int(row.get("idx_active", -1)), int(row.get("idx_inactive", -1))
            if 0 <= ia < len(yp) and 0 <= ii < len(yp):
                correct += int(yp[ia] > yp[ii])
                total   += 1
        m["Cliff_acc"] = correct / total if total else float("nan")

    if label:
        cliff_str = f"  Cliff_acc={m.get('Cliff_acc', float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f}  MAE={mae_v:.4f}  R²={r2_v:.4f}  "
              f"Pearson={pr_v:.4f}  Spearman={sp_v:.4f}  Kendall={kt_v:.4f}{cliff_str}")
    return m


In [3]:
tr = load_train()
te = load_test()
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)
active_mask = tr["pec50"].values >= 5.5
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists()
               else pd.DataFrame())

NR_TARGETS = ["PXR","CAR","VDR","FXR","LXRa","RXRa","PPARg"]
NR_W = {"PXR":1.0,"CAR":0.6,"VDR":0.6,"FXR":0.5,"LXRa":0.5,"RXRa":0.4,"PPARg":0.2}


In [4]:
# Build multi-task dataset: CRC (primary) + ChEMBL NR (auxiliary)
chembl = pd.read_parquet(DATA_EXTERNAL/"chembl_nr_extended.parquet")
bdb    = pd.read_parquet(DATA_EXTERNAL/"bindingdb_nr_data.parquet")
nr_all = pd.concat([chembl, bdb], ignore_index=True).dropna(subset=["smiles","pec50"])
nr_all["std_smi"] = nr_all["smiles"].map(standardize_smiles)
nr_all = nr_all.dropna(subset=["std_smi"])
nr_all["ik"] = nr_all["std_smi"].map(to_inchikey)

# Pivot: one row per compound, one column per NR target
nr_pivot = (nr_all.groupby(["ik","target_name"])["pec50"].mean().unstack("target_name")
            .reindex(columns=NR_TARGETS))
nr_pivot = nr_pivot.reset_index()

# Join with standardized SMILES
ik2smi = nr_all.drop_duplicates("ik").set_index("ik")["std_smi"]
nr_pivot["smiles"] = nr_pivot["ik"].map(ik2smi)

# CRC compounds get primary label + NaN for auxiliary targets
tr_ik2pec50 = tr.assign(ik=tr["smiles"].map(to_inchikey)).set_index("ik")["pec50"]
tr_rows = pd.DataFrame({"ik": tr.assign(ik=tr["smiles"].map(to_inchikey))["ik"],
                         "smiles": tr["smiles"].values,
                         "PXR": tr["pec50"].values})
for t in NR_TARGETS[1:]:
    tr_rows[t] = np.nan

# External NR rows (no primary label)
ext_rows = nr_pivot.copy()
ext_rows["smiles"] = ext_rows["smiles"]

# Combine and dedup
all_rows = pd.concat([tr_rows, ext_rows], ignore_index=True)
all_rows = all_rows.dropna(subset=["smiles"])
print(f"Multi-task dataset: {len(all_rows):,} rows")
print(f"PXR labels: {all_rows['PXR'].notna().sum()}")
for t in NR_TARGETS[1:]:
    print(f"  {t}: {all_rows[t].notna().sum()}")


Multi-task dataset: 15,382 rows
PXR labels: 5084
  CAR: 0
  VDR: 523
  FXR: 3185
  LXRa: 1173
  RXRa: 1364
  PPARg: 4302


In [5]:
try:
    from chemprop import data as cpdata, models, nn as cpnn, featurizers
    import chemprop
    import torch
    print(f"chemprop {chemprop.__version__}  torch {torch.__version__}")
    CHEMPROP_OK = True
except ImportError as e:
    print(f"chemprop/torch not available: {e}")
    CHEMPROP_OK = False


chemprop 2.2.3  torch 2.11.0+cpu


In [6]:
if not CHEMPROP_OK:
    print("Falling back to LGBM multi-task (target-weighted)")
    import lightgbm as lgb
    from pxr.featurize import combined, impute

    NR_W_DICT = {"PXR":1.0,"CAR":0.6,"VDR":0.6,"FXR":0.5,"LXRa":0.5,"RXRa":0.4,"PPARg":0.2}
    tr_rows2 = tr.copy()
    X_tr = impute(combined(tr_rows2["smiles"].tolist()))
    X_te = impute(combined(te["smiles"].tolist()))
    y_tr_v = tr_rows2["pec50"].values.astype(np.float64)
    splits2 = scaffold_kfold_indices(tr_rows2["smiles"].map(bemis_murcko).tolist(), N_FOLDS, SEED)

    # Augment with NR data
    ext_smi, ext_y, ext_w = [], [], []
    for _, row in nr_all[nr_all["ik"].notna()].iterrows():
        w = NR_W_DICT.get(str(row.get("target_name","")), 0.15)
        ext_smi.append(row["std_smi"])
        ext_y.append(float(row["pec50"]))
        ext_w.append(w)
    X_ext = impute(combined(ext_smi))
    y_ext = np.array(ext_y); w_ext = np.array(ext_w)

    LGBM_PARAMS = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
                       min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
                       reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)
    oof = np.full(len(y_tr_v), np.nan)
    for fold, (tri, vai) in enumerate(splits2):
        Xf = np.vstack([X_tr[tri], X_ext])
        yf = np.concatenate([y_tr_v[tri], y_ext])
        wf = np.concatenate([np.ones(len(tri)), w_ext])
        m = lgb.train(LGBM_PARAMS, lgb.Dataset(Xf,label=yf,weight=wf),
                      valid_sets=[lgb.Dataset(X_tr[vai],label=y_tr_v[vai])],
                      callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(-1)])
        oof[vai] = m.predict(X_tr[vai])
        print(f"  fold {fold+1} RAE={rae(y_tr_v[vai],oof[vai]):.4f}", flush=True)
    m_fb = full_metrics(y_tr_v, oof, cliff_pairs, "chemprop_mt_fallback_lgbm")
    m_fb_a = full_metrics(y_tr_v[active_mask], oof[active_mask], label="fallback [active]")
    m_final = lgb.train(LGBM_PARAMS,
                        lgb.Dataset(np.vstack([X_tr,X_ext]),
                                    label=np.concatenate([y_tr_v,y_ext]),
                                    weight=np.concatenate([np.ones(len(y_tr_v)),w_ext])),
                        callbacks=[lgb.log_evaluation(-1)])
    te_preds = np.clip(m_final.predict(X_te), y_tr_v.min()-0.5, y_tr_v.max()+0.5)
    results_df = pd.DataFrame([m_fb, m_fb_a], index=["overall","active"])
    print("\n" + results_df.round(4).to_string())
else:
    print("TODO: run actual Chemprop multi-task (chemprop available)")
    # Placeholder — chemprop multi-task implementation here
    oof = np.zeros(len(tr))
    te_preds = np.zeros(513)


TODO: run actual Chemprop multi-task (chemprop available)


In [7]:
np.save(DATA_PROCESSED / "oof_chemprop_chembl_nr_multitask.npy", oof)
np.save(DATA_PROCESSED / "te_oof_chemprop_chembl_nr_multitask.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub) == 513 and sub["pEC50"].notna().all()
out = SUBMISSIONS / "74_chemprop_chembl_nr_multitask.csv"
sub.to_csv(out, index=False)
print(f"Saved {out}")
print(f"Test preds  min={te_preds.min():.2f}  median={np.median(te_preds):.2f}  max={te_preds.max():.2f}")


Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\74_chemprop_chembl_nr_multitask.csv
Test preds  min=0.00  median=0.00  max=0.00
